# Exploratory Data Analysis (EDA)

This notebook performs exploratory data analysis on the phishing URL dataset.

## Dataset Information
- **Source**: Kaggle - Phishing Dataset for Machine Learning
- **Features**: 48 pre-extracted URL features
- **Labels**: CLASS_LABEL (0 = Legitimate, 1 = Phishing)
- **Samples**: 10,000 URLs (5,000 phishing, 5,000 legitimate)

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

# Paths
DATA_PATH = '../data/raw/phishing_dataset.csv'
FIGURES_DIR = '../reports/figures'

print("Libraries imported successfully.")

## 1. Load Dataset

In [ ]:
# Load the dataset
df = pd.read_csv(DATA_PATH)

# Display basic information
print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
df.head()

## 2. Dataset Summary Statistics

In [ ]:
# Basic statistics
print("="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"Shape: {df.shape}")
print(f"Features: {df.shape[1] - 1}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"\nLabel distribution:")
print(df['CLASS_LABEL'].value_counts())
print("\n" + "="*60)
print("DESCRIPTIVE STATISTICS")
print("="*60)
df.describe().T

## 3. Class Distribution

In [ ]:
# Class distribution pie chart
fig, ax = plt.subplots(figsize=(6, 6))
counts = df['CLASS_LABEL'].value_counts().sort_index()
labels = ['Legitimate', 'Phishing']
colors = ['#2ecc71', '#e74c3c']

wedges, texts, autotexts = ax.pie(
    counts.values,
    labels=labels,
    colors=colors,
    autopct="%1.1f%%",
    startangle=140,
    pctdistance=0.80,
    wedgeprops={"width": 0.55, "edgecolor": "white", "linewidth": 2},
)
for at in autotexts:
    at.set_fontsize(13)
    at.set_fontweight('bold')

ax.set_title('Class Distribution (Phishing vs. Legitimate)', pad=20, fontweight='bold')
ax.axis('equal')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/01_class_distribution_pie.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_class_distribution_pie.png")

In [ ]:
# Class distribution bar chart
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.2, width=0.5)

total = counts.sum()
for bar, count in zip(bars, counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total * 0.005,
        f"{count:,}\n({count / total:.1%})",
        ha='center', va='bottom', fontsize=11, fontweight='bold',
    )

ax.set_title('Class Distribution – Count', fontweight='bold')
ax.set_ylabel('Sample Count')
ax.set_ylim(0, max(counts.values) * 1.15)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/02_class_distribution_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 02_class_distribution_bar.png")

## 4. Correlation Heatmap (Top 20 Features)

In [ ]:
# Select top 20 features most correlated with target
numeric_df = df.select_dtypes(include=np.number)
correlations = numeric_df.corr()['CLASS_LABEL'].abs().sort_values(ascending=False)
top_features = correlations.iloc[1:21].index.tolist()
subset = df[top_features + ['CLASS_LABEL']]

corr_matrix = subset.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0,
    linewidths=0.4,
    vmin=-1, vmax=1,
    ax=ax,
    annot_kws={"size": 7},
)
ax.set_title('Correlation Heatmap – Top 20 Features', fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/03_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 03_correlation_heatmap.png")

## 5. Feature Distributions

In [ ]:
# Map column names
col_map = {c.lower(): c for c in df.columns}
features_to_plot = ['urllength', 'numdots']
found_features = [col_map[f] for f in features_to_plot if f in col_map]

if found_features:
    fig, axes = plt.subplots(1, len(found_features), figsize=(7 * len(found_features), 5))
    if len(found_features) == 1:
        axes = [axes]

    for ax, feat in zip(axes, found_features):
        for label_val, color in zip(sorted(df['CLASS_LABEL'].unique()), colors):
            subset = df[df['CLASS_LABEL'] == label_val][feat].dropna()
            ax.hist(
                subset,
                bins=40,
                alpha=0.65,
                color=color,
                label=labels[label_val],
                edgecolor='white',
                linewidth=0.4,
            )
        ax.set_xlabel(feat, fontweight='bold')
        ax.set_ylabel('Frequency')
        ax.set_title(f'Distribution of {feat}', fontweight='bold')
        ax.legend(framealpha=0.8)
        ax.spines[['top', 'right']].set_visible(False)

    fig.suptitle('Feature Distributions by Class', fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/04_feature_histograms.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 04_feature_histograms.png")
else:
    print("UrlLength / NumDots columns not found")

## 6. Boxplots by Class

In [ ]:
if found_features:
    df_plot = df.copy()
    df_plot['CLASS_LABEL_STR'] = df_plot['CLASS_LABEL'].map({1: 'Phishing', 0: 'Legitimate'})

    fig, axes = plt.subplots(1, len(found_features), figsize=(7 * len(found_features), 5))
    if len(found_features) == 1:
        axes = [axes]

    for ax, feat in zip(axes, found_features):
        sns.boxplot(
            data=df_plot,
            x='CLASS_LABEL_STR',
            y=feat,
            palette={'Legitimate': colors[0], 'Phishing': colors[1]},
            linewidth=1.2,
            flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.4},
            ax=ax,
        )
        ax.set_title(f'{feat} by Class', fontweight='bold')
        ax.set_xlabel('')
        ax.spines[['top', 'right']].set_visible(False)

    fig.suptitle('Boxplots: Phishing vs. Legitimate', fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/05_boxplots.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 05_boxplots.png")

## 7. Missing Values

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(10, 5))
if missing.empty:
    ax.text(
        0.5, 0.5,
        'No Missing Values Found ✓',
        ha='center', va='center',
        fontsize=16, color='green', fontweight='bold',
        transform=ax.transAxes,
    )
else:
    missing.plot(kind='bar', ax=ax, color='#3498db', edgecolor='white')
    ax.set_ylabel('Missing Count')
    ax.set_title('Missing Values per Feature', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/06_missing_value_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 06_missing_value_matrix.png")

## 8. Feature Importance

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

X = df.drop(columns=['CLASS_LABEL']).select_dtypes(include=np.number)
y = df['CLASS_LABEL'].values
feature_names = X.columns.tolist()

clf = ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(X.values, y)

importances = pd.Series(clf.feature_importances_, index=feature_names)
importances = importances.nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(9, 8))
colors_imp = sns.color_palette("YlOrRd", len(importances))
importances.plot(kind='barh', ax=ax, color=colors_imp, edgecolor='white')
ax.set_title('Top 20 Feature Importances (ExtraTrees)', fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity')
ax.spines[['top', 'right']].set_visible(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/07_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 07_feature_importance.png")

## 9. Pairplot (Top 5 Features)

In [ ]:
# Get top 5 features by correlation
numeric_df = df.select_dtypes(include=np.number)
correlations = numeric_df.corr()['CLASS_LABEL'].abs().sort_values(ascending=False)
top_features = correlations.iloc[1:6].index.tolist()

df_plot = df[top_features + ['CLASS_LABEL']].copy()
df_plot['CLASS_LABEL_STR'] = df_plot['CLASS_LABEL'].map({1: 'Phishing', 0: 'Legitimate'})

g = sns.pairplot(
    df_plot,
    hue='CLASS_LABEL_STR',
    vars=top_features,
    palette={'Legitimate': colors[0], 'Phishing': colors[1]},
    diag_kind='kde',
    plot_kws={'alpha': 0.4, 's': 15},
    corner=True,
)
g.fig.suptitle('Pairplot – Top 5 Discriminating Features', y=1.02, fontweight='bold')
plt.savefig(f'{FIGURES_DIR}/08_pairplot_top_features.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 08_pairplot_top_features.png")

## Summary

EDA complete! All visualizations have been saved to the `reports/figures/` directory:

1. Class distribution pie chart
2. Class distribution bar chart
3. Correlation heatmap (top 20 features)
4. Feature histograms
5. Boxplots by class
6. Missing value analysis
7. Feature importance ranking
8. Pairplot of top 5 features